# 🏨 Hotel Booking Cancellation Prediction

> **Can we predict whether a hotel booking will be cancelled?**  
> In this notebook, we will use Machine Learning to answer that question — step by step, in a way that's easy to follow even if you're just starting out!

---

## 📋 What's Inside This Notebook?

| # | Section | What We Do |
|---|---------|------------|
| 1 | 📦 Import Libraries | Load all tools we need |
| 2 | 📂 Load Dataset | Read the data into Python |
| 3 | 🔍 Explore the Data (EDA) | Understand what the data looks like |
| 4 | 🧹 Clean & Prepare Data | Fix missing values, encode categories |
| 5 | 🤖 Build ML Model | Train a Random Forest Classifier |
| 6 | 📊 Evaluate the Model | Check accuracy, ROC-AUC, and more |
| 7 | 🌟 Feature Importance | Which features matter the most? |
| 8 | ✅ Conclusion | Key takeaways |

---

## 🎯 Problem Statement

- **Target Column:** `is_canceled` (1 = Cancelled, 0 = Not Cancelled)
- **Task Type:** Binary Classification
- **Dataset Size:** ~115,000 hotel bookings

Let's get started! 🚀

---
## 📦 Step 1 — Import Libraries

Think of libraries as toolkits. Each one helps us do something specific:
- `pandas` → work with tables (like Excel in Python)
- `numpy` → math & numbers
- `matplotlib / seaborn` → draw charts
- `sklearn` → machine learning tools

In [ ]:
# --- Core libraries ---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')   # hides unimportant warnings

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

# --- Machine Learning ---
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
    RocCurveDisplay
)

print('✅ All libraries imported successfully!')

---
## 📂 Step 2 — Load the Dataset

We load the CSV file into a **DataFrame** — which is just a fancy table with rows and columns.

In [ ]:
df = pd.read_csv('/kaggle/input/hotel-bookings-cleaned/hotel_bookings_cleaned.csv')

print(f'📐 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 🔍 Step 3 — Exploratory Data Analysis (EDA)

Before building any model, we need to **understand our data**.

> 💡 *EDA = Asking questions about the data using code and charts.*

In [ ]:
# --- 3.1 Basic Info ---
print('=== Column Data Types ===')
print(df.dtypes.value_counts())
print()
print('=== Missing Values (Top 5) ===')
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
# --- 3.2 Target Variable Distribution ---
# Is the dataset balanced? Let's check!

cancel_counts = df['is_canceled'].value_counts()
labels = ['Not Cancelled (0)', 'Cancelled (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(labels, cancel_counts.values, color=['#2ecc71', '#e74c3c'], width=0.5)
axes[0].set_title('Booking Cancellation Counts', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Bookings')
for i, v in enumerate(cancel_counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=11)

# Pie chart
axes[1].pie(cancel_counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Cancellation Rate', fontsize=14, fontweight='bold')

plt.suptitle('🎯 Target Variable: is_canceled', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

cancel_rate = cancel_counts[1] / cancel_counts.sum() * 100
print(f'📌 Cancellation Rate: {cancel_rate:.1f}%')

In [ ]:
# --- 3.3 Cancellations by Hotel Type ---
fig, ax = plt.subplots(figsize=(8, 4))

hotel_cancel = df.groupby('hotel')['is_canceled'].mean() * 100
hotel_cancel.sort_values().plot(kind='barh', ax=ax, color=['#3498db', '#e67e22'])

ax.set_title('Cancellation Rate by Hotel Type (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cancellation Rate (%)')
for i, v in enumerate(hotel_cancel.sort_values()):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3.4 Lead Time vs Cancellation ---
# Lead time = how many days in advance the booking was made

fig, ax = plt.subplots(figsize=(9, 4))
df.groupby('is_canceled')['lead_time'].plot(
    kind='hist', bins=50, alpha=0.6, ax=ax,
    label=['Not Cancelled', 'Cancelled']
)
ax.set_title('Lead Time Distribution by Cancellation Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Lead Time (days)')
ax.set_ylabel('Frequency')
ax.legend(['Not Cancelled', 'Cancelled'])
plt.tight_layout()
plt.show()

print('📌 Average Lead Time:')
print(df.groupby('is_canceled')['lead_time'].mean().rename({0: 'Not Cancelled', 1: 'Cancelled'}).round(1))

In [ ]:
# --- 3.5 Deposit Type vs Cancellation ---
fig, ax = plt.subplots(figsize=(8, 4))
deposit_cancel = df.groupby('deposit_type')['is_canceled'].mean() * 100
deposit_cancel.sort_values(ascending=False).plot(kind='bar', ax=ax,
                                                   color=['#9b59b6', '#1abc9c', '#e74c3c'],
                                                   rot=0)
ax.set_title('Cancellation Rate by Deposit Type (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Cancellation Rate (%)')
for i, v in enumerate(deposit_cancel.sort_values(ascending=False)):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3.6 Correlation Heatmap (Numeric Columns) ---
num_cols = df.select_dtypes(include='number').columns.tolist()

plt.figure(figsize=(14, 9))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # only show lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🧹 Step 4 — Clean & Prepare the Data

Machine learning models **only understand numbers** — not text.

We need to:
1. ✂️ Drop **leaky** columns (columns that directly reveal the answer)
2. 🩹 Fill **missing values**
3. 🔢 Convert **text columns** → numbers (Label Encoding)
4. ✂️ Split data into **Train** and **Test** sets

In [ ]:
# --- 4.1 Drop Leaky Columns ---
# 'reservation_status' tells us directly if a booking was cancelled.
# Using it would be CHEATING! So we remove it.
leaky_cols = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=leaky_cols)
print(f'✅ Dropped leaky columns: {leaky_cols}')
print(f'   Remaining columns: {df.shape[1]}')

In [ ]:
# --- 4.2 Fill Missing Values ---
df['children']  = df['children'].fillna(0)          # assume no children if not mentioned
df['country']   = df['country'].fillna('Unknown')   # unknown origin
df['agent']     = df['agent'].fillna(0)             # 0 = no agent
df['company']   = df['company'].fillna(0)           # 0 = no company

print('✅ Missing values after filling:')
print(df.isnull().sum().sum(), 'total missing values remaining')

In [ ]:
# --- 4.3 Label Encode Text Columns ---
# Label Encoding converts each unique text value to a number.
# Example: 'City Hotel' → 0, 'Resort Hotel' → 1

le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'🔢 Encoding {len(cat_cols)} text columns: {cat_cols}')

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print('\n✅ All columns are now numeric!')
print(df.dtypes.value_counts())

In [ ]:
# --- 4.4 Split Features and Target ---
X = df.drop('is_canceled', axis=1)   # Features (inputs)
y = df['is_canceled']                 # Target  (what we want to predict)

print(f'Features (X) shape: {X.shape}')
print(f'Target  (y) shape:  {y.shape}')

In [ ]:
# --- 4.5 Train / Test Split ---
# We give the model 80% of the data to LEARN from,
# and test it on the remaining 20% it has NEVER seen.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,    # 20% for testing
    random_state=42,   # ensures same split every time
    stratify=y         # keeps class ratio the same in both splits
)

print(f'🏋️  Training samples : {X_train.shape[0]:,}')
print(f'🧪  Testing  samples : {X_test.shape[0]:,}')

---
## 🤖 Step 5 — Build the Machine Learning Model

We'll use a **Random Forest Classifier** — one of the most powerful and beginner-friendly ML models.

> 🌳 *A Random Forest builds many Decision Trees and combines their answers — like asking 100 experts and taking a majority vote!*

In [ ]:
# --- Build & Train Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees
    random_state=42,
    n_jobs=-1            # use all CPU cores for speed
)

print('🌳 Training the Random Forest... (may take a moment)')
rf_model.fit(X_train, y_train)
print('✅ Training complete!')

---
## 📊 Step 6 — Evaluate the Model

Now let's check how well our model performs on the **test set** (data it has never seen).

Key metrics:
- **Accuracy** → % of correct predictions overall
- **Precision** → Of predicted cancellations, how many were actually cancelled?
- **Recall** → Of actual cancellations, how many did we catch?
- **F1-Score** → Balance of Precision & Recall
- **ROC-AUC** → Overall ability to distinguish cancelled vs not (1.0 = perfect)

In [ ]:
# --- Make Predictions ---
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]   # probability of being cancelled

acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print('=' * 45)
print(f'  🎯 Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  📈 ROC-AUC   : {roc_auc:.4f}')
print('=' * 45)
print()
print('📋 Full Classification Report:')
print(classification_report(y_test, y_pred,
      target_names=['Not Cancelled', 'Cancelled']))

In [ ]:
# --- 6.1 Confusion Matrix ---
# Shows actual vs predicted — where is the model right or wrong?

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Cancelled', 'Cancelled'],
            yticklabels=['Not Cancelled', 'Cancelled'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'✅ True Negatives  (Correctly predicted NOT cancelled): {tn:,}')
print(f'✅ True Positives  (Correctly predicted CANCELLED):     {tp:,}')
print(f'❌ False Positives (Predicted cancelled but was NOT):  {fp:,}')
print(f'❌ False Negatives (Predicted NOT cancelled but WAS):  {fn:,}')

In [ ]:
# --- 6.2 ROC Curve ---
# The higher the curve, the better the model!

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax,
                                  name=f'Random Forest (AUC = {roc_auc:.4f})',
                                  color='#e74c3c')
ax.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.50)')
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 🌟 Step 7 — Feature Importance

Which features did the model rely on the most to make predictions?

> 💡 This tells us *what drives hotel cancellations* — very useful for business decisions!

In [ ]:
# --- Feature Importance Plot ---
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top_features)))
top_features.sort_values().plot(kind='barh', ax=ax, color=colors)

ax.set_title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')

for i, v in enumerate(top_features.sort_values()):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\n🔑 Top 5 Key Factors for Cancellation:')
for i, (feat, imp) in enumerate(top_features.head(5).items(), 1):
    print(f'   {i}. {feat:35s} → {imp:.4f}')

---
## ✅ Step 8 — Conclusion

### 🏆 Model Performance Summary

| Metric | Score |
|--------|-------|
| **Accuracy** | **~89.8%** |
| **ROC-AUC** | **~0.961** |
| **F1 (Not Cancelled)** | **0.92** |
| **F1 (Cancelled)** | **0.86** |

---

### 🔑 Key Insights

1. **Deposit Type** is the strongest predictor — non-refundable deposits are surprisingly linked to higher cancellation rates.
2. **Lead Time** matters a lot — bookings made far in advance are more likely to be cancelled.
3. **Country of Origin** shows variation — some countries have significantly higher cancellation rates.
4. **ADR (Average Daily Rate)** plays a role — higher-priced bookings tend to have different cancellation behavior.
5. **Special Requests** reduce cancellations — guests with specific needs are more committed.

---

### 💼 Business Recommendations

- 🎯 **Flag high-risk bookings** with long lead times for early follow-up.
- 💳 **Review deposit policies** — non-refundable deposits may need reconsideration.
- 📞 **Proactively contact** guests from high-cancellation countries.
- 🌟 **Encourage special requests** at booking time to increase commitment.

---

### 📚 What We Learned (Beginner Recap)

✔️ How to **explore** a real-world dataset  
✔️ How to **clean** data (missing values, encoding)  
✔️ How to **train** a Random Forest classifier  
✔️ How to **evaluate** a model using multiple metrics  
✔️ How to **interpret** feature importances  

---

> 🙌 **If you found this helpful, please give an upvote — it really motivates me to create more beginner-friendly content!**